# Complaint Triage System: training IndoBERT di Google Colab

1. **Runtime → Change runtime type → T4 GPU → Save.**
2. Jalankan sel dari atas ke bawah (**Shift+Enter**).
3. Di sel ke-3 kamu diminta mengunggah `data/annotated/brimo.jsonl` dari laptop (hasil `labelstudio.py silver`).

Notebook ini hanya memanggil skrip di repo, jadi hasilnya sama dengan menjalankan skrip di laptop.
Torch bawaan Colab dipakai apa adanya (sudah versi GPU), jadi tidak di-install ulang.

In [ ]:
# 1) Ambil kode dan pasang library yang dibutuhkan (sekitar 1 menit)
!git clone -q https://github.com/HachiVin/Complaint-Triage-System.git
%cd Complaint-Triage-System
!pip install -q transformers==5.20.2 tokenizers==0.23.3 scikit-learn==1.9.1 joblib==1.6.0 pydantic==2.14.1
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA - ganti runtime ke T4 GPU dulu")

In [ ]:
# 2) Cek: model baseline v1 dan catatan label sudah ikut ter-push dari laptop?
import json, os
meta = json.load(open("models/baseline/model_meta.json"))
print("baseline version:", meta.get("version"), "| train_size:", meta.get("train_size"))
print("label_provenance:", "ada" if os.path.exists("docs/label_provenance.json") else "TIDAK ADA - push dulu dari laptop")
if meta.get("version") == "demo-synthetic":
    print("PERINGATAN: ini masih model demo. Jalankan train_baseline.py di laptop lalu push dulu.")

In [ ]:
# 3) Unggah data/annotated/brimo.jsonl dari laptop (sekitar 15 MB)
import shutil
from google.colab import files
os.makedirs("data/annotated", exist_ok=True)
for name in files.upload():
    shutil.move(name, f"data/annotated/{name}")
!ls -la data/annotated

In [ ]:
# 4) Split per waktu, sama seperti di laptop: 15% ulasan terbaru jadi test set
!python scripts/prepare_data.py data/annotated/brimo.jsonl data/processed

In [ ]:
# 5) Fine-tune IndoBERT multi-task (aspek + sentimen + urgensi), sekitar 10-25 menit di T4.
# Progres muncul tiap 200 step. Jangan tutup tab ini selama training.
!python scripts/train_multitask.py data/processed models/indobert-mt --epochs 3

In [ ]:
# 6) Bandingkan aturan, TF-IDF, dan IndoBERT di test set -> docs/results.md
!python scripts/evaluate.py data/processed --baseline models/baseline --multitask models/indobert-mt
print(open("docs/results.md").read())

In [ ]:
# 7) Unduh model IndoBERT dan hasil evaluasi (sekitar 450 MB)
!zip -qr indobert-mt.zip models/indobert-mt docs/results.md docs/results.json
files.download("indobert-mt.zip")